In [1]:
import requests
import torch
from PIL import Image
from transformers import MllamaForConditionalGeneration, AutoProcessor
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"


model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)



Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [2]:
all_files = []
img_dir = """/projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped"""
for path, subdirs, files in os.walk(img_dir):
    for name in files:
        all_files.append(os.path.join(path, name))

In [3]:
# """/projects/matsci/vlm_microscopy/Microscopy/BBBC005/sampled_images/w1_w2/images"""

In [4]:
len(all_files)

51

In [5]:
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "This is an SEM image of cells. Please count the number of cells in this image in the format 'Count: <count>'. If you are unable to count the cells, please write NaN."}
    ]}
]

In [6]:
all_files[0]

'/projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped/Particles/L2_0a9d325c8250edd1e89670d17b0b04ec.jpg'

In [7]:
from tqdm import tqdm

responses = []
actuals = []
for img_path in tqdm(all_files, desc="Processing items"):
    image = Image.open(img_path)
    # ground_truth = folder_id_map[img_path.split("/")[-2]]
    # actuals.append(ground_truth)

    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)
    input_length = inputs.input_ids.shape[1]

    output = model.generate(**inputs, max_new_tokens=50, temperature=0.2, top_k=5)
    res = processor.decode(output[0][input_length:], skip_special_tokens=True)
    responses.append(res)
    # print(img_path)
    # print(res)
    

Processing items: 100%|█████████████████████████████████████████████████████████████| 51/51 [00:17<00:00,  2.96it/s]


In [9]:
# actuals = []
# for file in all_files:
#     actuals.append(int(file.split("_")[-4][1:]))

In [10]:
set(responses)

{'Count: 1',
 'Count: 1.',
 'Count: 10',
 'Count: 12',
 'Count: 12.',
 'Count: 16.',
 'Count: 17',
 'Count: 2',
 'Count: 2.',
 'Count: 23',
 'Count: 25.',
 'Count: 26',
 'Count: 3',
 'Count: 3.',
 'Count: 38',
 'Count: 4',
 'Count: 4.',
 'Count: 5',
 'Count: 5.',
 'Count: 53',
 'Count: 6',
 'Count: 63.',
 'Count: 7',
 'Count: 7.',
 'Count: 8',
 'Count: 9',
 'The image shows a total of 3 fibres or particles.'}

In [11]:
import re

predictions = []
for idx, path in enumerate(all_files):
    numbers = re.findall(r'\d+', responses[idx])
    pred = -1
    if len(numbers) > 0:
        pred = int(numbers[0])
    predictions.append(pred)

In [12]:
import pandas as pd
df = pd.DataFrame({'img_path': all_files, 'raw_predictions': responses,  'predictions': predictions})

In [13]:
df.to_csv('counting_NFFA_llama_manually_sampled.csv')

In [14]:
from collections import Counter

In [15]:
elem_counts = Counter(responses)

In [16]:
print(elem_counts)

Counter({'Count: 7.': 5, 'Count: 9': 4, 'Count: 1': 4, 'Count: 5': 4, 'Count: 4': 3, 'Count: 2': 3, 'Count: 25.': 2, 'Count: 7': 2, 'Count: 8': 2, 'Count: 2.': 2, 'Count: 5.': 2, 'Count: 6': 2, 'Count: 3': 2, 'Count: 38': 1, 'Count: 16.': 1, 'Count: 26': 1, 'Count: 23': 1, 'Count: 63.': 1, 'Count: 1.': 1, 'The image shows a total of 3 fibres or particles.': 1, 'Count: 12': 1, 'Count: 53': 1, 'Count: 17': 1, 'Count: 12.': 1, 'Count: 4.': 1, 'Count: 10': 1, 'Count: 3.': 1})
